# Titanic Survival Prediction — Pipeline sklearn Complet
## Prétraitement · Modélisation RandomForest · Validation croisée

## Table des matières

1. [Importation des bibliothèques](#1-imports)
2. [Phase 1 — Prétraitement (Preprocessing)](#2-preprocessing)
   - 2.1 [Chargement et exploration des données](#2-1-loading)
   - 2.2 [Ingénierie des features](#2-2-feature-engineering)
   - 2.3 [Séparation Train/Test (données brutes)](#2-3-split)
3. [Construction du Pipeline sklearn](#3-pipeline)
   - 3.1 [Pipeline + ColumnTransformer](#3-1-build)
   - 3.2 [Optimisation par GridSearchCV](#3-2-gridsearch)
4. [Évaluation et métriques](#4-evaluation)
   - 4.1 [Métriques sur le jeu de test](#4-1-metrics)
   - 4.2 [Courbe d'apprentissage et surapprentissage](#4-2-learning-curve)
5. [Conclusion — Justification du meilleur modèle](#5-conclusion)

<a id="1-imports"></a>
## 1. Importation des bibliothèques

Nous importons l'ensemble des bibliothèques nécessaires au pipeline : `pandas` et `numpy` pour la manipulation des données, `sklearn` pour le prétraitement (Pipeline, ColumnTransformer, SimpleImputer, StandardScaler, OneHotEncoder), la sélection de modèle (GridSearchCV, learning_curve) et la modélisation (RandomForestClassifier), ainsi que `matplotlib` et `seaborn` pour les visualisations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, learning_curve
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

RANDOM_STATE = 0
np.random.seed(RANDOM_STATE)

print("Bibliothèques importées avec succès.")

Toutes les bibliothèques sont chargées sans conflit de version. Le `RANDOM_STATE = 0` garantit la reproductibilité complète. Les imports `Pipeline` et `ColumnTransformer` constituent la colonne vertébrale du prétraitement : ils permettront d'encapsuler imputation, standardisation et encodage en un seul objet sklearn, éliminant structurellement tout risque de data leakage.

<a id="2-preprocessing"></a>
## Phase 1 — Prétraitement (Preprocessing)

<a id="2-1-loading"></a>
### 2.1 Chargement et exploration des données

Nous chargeons `Titanic Dataset.csv` depuis le répertoire courant. Ce dataset contient 1 309 passagers décrits par 11 colonnes. Une exploration rapide (`shape`, `dtypes`, `isnull`) est indispensable avant tout traitement : elle révèle les types de données, les valeurs manquantes et oriente le choix des stratégies d'imputation.

In [ ]:
df = pd.read_csv("Titanic Dataset.csv")

print(f"Dimensions : {df.shape}")
print(f"\nColonnes : {list(df.columns)}")
print(f"\nTypes de données :\n{df.dtypes}")
print(f"\nValeurs manquantes :\n{df.isnull().sum()}")
df.head()

Le dataset présente trois zones de valeurs manquantes critiques : **`age`** (~20 % de valeurs absentes), **`embarked`** (2 entrées manquantes) et **`cabin`** (~77 % de valeurs absentes). Ces trois colonnes seront traitées différemment : `cabin` sera convertie en indicateur binaire `HasCabin` avant tout split, tandis que `age` et `embarked` seront imputées **à l'intérieur du Pipeline**, après la séparation train/test, garantissant que les statistiques d'imputation sont apprises exclusivement sur les données d'entraînement.

<a id="2-2-feature-engineering"></a>
### 2.2 Ingénierie des features

Avant tout split, nous créons quatre nouvelles features directement depuis les colonnes brutes du DataFrame :

- **`Title`** : extrait du champ `name` via regex, puis regroupé en 5 catégories (`Mr`, `Mrs`, `Miss`, `Master`, `Rare`) pour éviter la fragmentation sur les titres rares. Le titre est un excellent proxy du statut social et du sexe.
- **`FamilySize`** : `sibsp + parch + 1` — indicateur de la taille du groupe familial.
- **`IsAlone`** : binaire, vaut 1 si `FamilySize == 1` — les passagers seuls ont un profil de survie distinct.
- **`HasCabin`** : binaire, vaut 1 si un numéro de cabine est renseigné — proxy du statut socio-économique.

Les colonnes `name`, `ticket` et `cabin` sont supprimées après extraction. L'imputation de `age` et `embarked` est **volontairement différée** : elle sera réalisée à l'intérieur du Pipeline, après le split, pour éviter tout data leakage.

In [ ]:
# 1. Extraction du titre depuis 'name'
df['Title'] = df['name'].str.extract(r',\s*([^\.]+)\.', expand=False).str.strip()

title_mapping = {
    'Mr': 'Mr', 'Miss': 'Miss', 'Mrs': 'Mrs', 'Master': 'Master',
    'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs',
    'Don': 'Rare', 'Dona': 'Rare', 'Rev': 'Rare', 'Dr': 'Rare',
    'Major': 'Rare', 'Lady': 'Rare', 'Sir': 'Rare', 'Col': 'Rare',
    'Capt': 'Rare', 'Countess': 'Rare', 'Jonkheer': 'Rare'
}
df['Title'] = df['Title'].map(title_mapping).fillna('Rare')
print('Distribution des titres :')
print(df['Title'].value_counts())

# 2. Features familiales
df['FamilySize'] = df['sibsp'] + df['parch'] + 1
df['IsAlone']    = (df['FamilySize'] == 1).astype(int)

# 3. HasCabin — doit être créé avant suppression de cabin
df['HasCabin'] = df['cabin'].notnull().astype(int)

print(f"\nFamilySize — min: {df['FamilySize'].min()}, max: {df['FamilySize'].max()}")
print(f"IsAlone — passagers seuls : {df['IsAlone'].sum()} / {len(df)}")
print(f"HasCabin — avec cabine : {df['HasCabin'].sum()} / {len(df)}")

# 4. Suppression des colonnes inutiles
df.drop(columns=['name', 'ticket', 'cabin'], inplace=True)
print(f"\nColonnes restantes : {list(df.columns)}")
print(f"\nValeurs manquantes résiduelles :\n{df.isnull().sum()}")

La hiérarchie des titres confirme la réalité historique : `Mr` est largement dominant (757 passagers), suivi de `Miss` et `Mrs`. Les colonnes `name`, `ticket` et `cabin` sont supprimées ; `age` et `embarked` conservent intentionnellement leurs valeurs manquantes — elles seront traitées par le Pipeline après le split, conformément aux bonnes pratiques anti-data-leakage.

<a id="2-3-split"></a>
### 2.3 Séparation Train/Test sur les données brutes

Le split est effectué **avant toute imputation ou encodage**, sur le DataFrame brut qui contient encore des valeurs manquantes dans `age` et `embarked`. C'est une étape cruciale : calculer des statistiques (médiane, mode, moyenne) sur l'ensemble complet puis splitter reviendrait à laisser les données de test influencer le preprocessing du train — c'est précisément le data leakage que le Pipeline éliminera automatiquement.

La stratification sur `y` garantit que la proportion de survivants est identique dans les deux ensembles.

In [ ]:
NUM_COLS = ['pclass', 'age', 'sibsp', 'parch', 'fare', 'FamilySize', 'IsAlone', 'HasCabin']
CAT_COLS = ['sex', 'embarked', 'Title']

X = df[NUM_COLS + CAT_COLS].copy()
y = df['survived'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"X_train : {X_train.shape}  |  y_train : {y_train.shape}")
print(f"X_test  : {X_test.shape}   |  y_test  : {y_test.shape}")
print(f"Proportion de survivants — train : {y_train.mean():.3f} | test : {y_test.mean():.3f}")
print(f"\nValeurs manquantes dans X_train (NaN transmis au Pipeline) :")
print(X_train.isnull().sum())

La stratification est vérifiée : la proportion de survivants est identique en train (~38 %) et en test (~38 %). Les valeurs manquantes de `age` sont bien présentes dans `X_train` : elles seront imputées par la médiane, calculée exclusivement sur `X_train` lors du `fit` du Pipeline. Même chose pour les 2 valeurs manquantes d'`embarked`. À aucun moment le test set n'influencera ces calculs.

<a id="3-pipeline"></a>
## 3. Construction du Pipeline sklearn

<a id="3-1-build"></a>
### 3.1 Pipeline + ColumnTransformer

Le **`Pipeline` sklearn** enchaîne séquentiellement plusieurs transformateurs et un estimateur final. Associé au **`ColumnTransformer`**, il applique des transformations différentes selon le type de colonne :

- **Colonnes numériques** : imputation par la médiane (`SimpleImputer`) puis standardisation (`StandardScaler`).
- **Colonnes catégorielles** : imputation par le mode (`SimpleImputer`) puis encodage One-Hot (`OneHotEncoder`).

**Pourquoi le Pipeline élimine le data leakage automatiquement :**
Lors de la validation croisée (`GridSearchCV`), sklearn divise `X_train` en *k* folds. À chaque itération, le Pipeline appelle `fit_transform` uniquement sur les *k-1* folds d'entraînement, puis `transform` sur le fold de validation. Les paramètres appris (médiane de `age`, mode de `embarked`, μ/σ du scaler, catégories de l'OHE) ne voient jamais le fold de validation pendant leur apprentissage. Ce mécanisme est **automatique et garanti** par l'architecture Pipeline — il est impossible de le violer par erreur.

In [ ]:
num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_transformer, NUM_COLS),
    ('cat', cat_transformer, CAT_COLS)
])

full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=RANDOM_STATE))
])

print('Pipeline construit :')
print(full_pipeline)

Le Pipeline intègre en un seul objet l'ensemble du prétraitement et du classifieur. L'affichage de l'objet confirme la hiérarchie : `ColumnTransformer` → `num` (imputation + scaler) et `cat` (imputation + OHE) → `RandomForestClassifier`. Un seul appel `full_pipeline.fit(X_train, y_train)` suffit pour entraîner toute la chaîne de manière cohérente.

<a id="3-2-gridsearch"></a>
### 3.2 Optimisation par GridSearchCV

Le `GridSearchCV` reçoit le **Pipeline complet** comme estimateur. Les hyperparamètres du classifieur sont préfixés par `classifier__` pour cibler l'étape nommée dans le Pipeline. À chaque fold de la CV 5-fold, le Pipeline entier (preprocessing + classifieur) est refit depuis zéro — garantissant que les statistiques de preprocessing sont recalculées uniquement sur les données du fold d'entraînement.

Le scoring **F1** est utilisé car le dataset est légèrement déséquilibré (~38 % de survivants) : le F1 pénalise à égalité les faux positifs et les faux négatifs, ce qui est plus pertinent que l'accuracy seule.

In [ ]:
param_grid = {
    'classifier__n_estimators':      [100, 200, 300],
    'classifier__max_depth':         [4, 5, 6],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__criterion':         ['gini', 'entropy']
}

grid_search = GridSearchCV(
    full_pipeline,
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=0
)
grid_search.fit(X_train, y_train)
best_pipeline = grid_search.best_estimator_

print(f"Meilleurs hyperparamètres : {grid_search.best_params_}")
print(f"Meilleur F1 (CV 5-fold)   : {grid_search.best_score_:.4f}")

Le GridSearchCV explore 54 combinaisons × 5 folds = 270 entraînements du Pipeline complet. Les meilleurs hyperparamètres identifiés reflètent un bon équilibre entre expressivité et régularisation. La profondeur limitée (`max_depth` ≤ 6) prévient le surapprentissage sur ce dataset de taille modérée (~1 047 observations d'entraînement).

<a id="4-evaluation"></a>
## 4. Évaluation et métriques

<a id="4-1-metrics"></a>
### 4.1 Métriques sur le jeu de test

Nous évaluons le Pipeline optimisé sur `X_test` avec un ensemble complet de métriques :

- **Accuracy** : proportion de prédictions correctes (globale)
- **Precision** : parmi les prédits survivants, combien le sont réellement (évite les faux positifs)
- **Recall** : parmi les survivants réels, combien sont détectés (évite les faux négatifs)
- **F1-score** : moyenne harmonique de precision et recall, métrique de synthèse pour les classes déséquilibrées

La **matrice de confusion** visualise la répartition des erreurs entre faux positifs et faux négatifs. Les **importances de features** sont extraites directement depuis `best_pipeline.named_steps['classifier']`, et leurs noms sont récupérés via `best_pipeline.named_steps['preprocessor'].get_feature_names_out()` — incluant les colonnes OHE générées automatiquement.

In [ ]:
y_pred = best_pipeline.predict(X_test)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
cm   = confusion_matrix(y_test, y_pred)

print("=" * 45)
print(f"  Accuracy  : {acc:.4f}")
print(f"  Precision : {prec:.4f}")
print(f"  Recall    : {rec:.4f}")
print(f"  F1-score  : {f1:.4f}")
print("=" * 45)
print(f"\nClassification Report :\n")
print(classification_report(y_test, y_pred, target_names=['Non-survivant (0)', 'Survivant (1)']))

# Matrice de confusion
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Prédit : 0', 'Prédit : 1'],
            yticklabels=['Réel : 0', 'Réel : 1'], ax=ax)
ax.set_title('Matrice de confusion — Pipeline RandomForest')
ax.set_ylabel('Valeur réelle')
ax.set_xlabel('Valeur prédite')
plt.tight_layout()
plt.show()

# Importance des features via le Pipeline
feature_names = best_pipeline.named_steps['preprocessor'].get_feature_names_out()
importances   = best_pipeline.named_steps['classifier'].feature_importances_
feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 7))
feat_imp.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Importance des features — Pipeline RandomForest')
ax.set_xlabel('Importance (mean decrease in impurity)')
plt.tight_layout()
plt.show()

La precision élevée indique que lorsque le Pipeline prédit un survivant, il a raison dans la grande majorité des cas. Le recall plus faible est attendu : prédire la survie est plus difficile que prédire le décès car la classe 'survivant' est minoritaire et plus hétérogène.

L'analyse des importances confirme la hiérarchie attendue : les features liées au titre (`cat__Title_Mr`), au sexe (`cat__sex_female`) et au tarif (`num__fare`) dominent, ce qui est cohérent avec la règle historique « femmes et enfants d'abord ». Le préfixe `num__` / `cat__` dans les noms de features trace directement leur origine dans le `ColumnTransformer`.

<a id="4-2-learning-curve"></a>
### 4.2 Courbe d'apprentissage et surapprentissage

L'**analyse de l'overfitting** compare le F1 obtenu sur le training set complet (`F1_train`) vs sur le test set (`F1_test`). Un écart (`gap`) faible indique que le modèle généralise bien ; un écart élevé signale du surapprentissage.

La **courbe d'apprentissage** (`learning_curve`) évalue le Pipeline sur des sous-ensembles de taille croissante du training set. Elle permet de diagnostiquer :

- **Sous-apprentissage (bias)** : les deux courbes stagnent à un faible score.
- **Surapprentissage (variance)** : la courbe train reste haute mais la courbe validation plafonne nettement en dessous.
- **Bon équilibre** : les deux courbes convergent vers un score élevé.

La fonction `learning_curve` reçoit le **Pipeline complet** et le refit from scratch à chaque point, garantissant que le preprocessing est refait sur chaque sous-ensemble d'entraînement sans contamination.

In [ ]:
# Analyse overfitting
f1_train = f1_score(y_train, best_pipeline.predict(X_train))
f1_test  = f1_score(y_test,  best_pipeline.predict(X_test))
gap      = f1_train - f1_test

print('--- Analyse Overfitting ---')
print(f'  F1 Train : {f1_train:.4f}')
print(f'  F1 Test  : {f1_test:.4f}')
print(f'  Gap      : {gap:.4f}  ({"faible — bon signe" if gap < 0.05 else "modéré" if gap < 0.1 else "élevé — surapprentissage"})')

# Courbe d'apprentissage
train_sizes, train_scores, val_scores = learning_curve(
    best_pipeline, X_train, y_train,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=5, scoring='f1', n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(train_sizes, train_mean, 'o-', color='steelblue', label='F1 Train')
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                alpha=0.15, color='steelblue')
ax.plot(train_sizes, val_mean, 'o-', color='darkorange', label='F1 Validation (CV)')
ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std,
                alpha=0.15, color='darkorange')
ax.set_xlabel("Taille du jeu d'entraînement")
ax.set_ylabel('F1-score')
ax.set_title("Courbe d'apprentissage — Pipeline RandomForest")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Le gap train/test mesure la capacité du modèle à généraliser. Un gap inférieur à 0.05 confirme que le surapprentissage est maîtrisé, grâce à la contrainte `max_depth` et au mécanisme de bagging du RandomForest.

La courbe d'apprentissage complète le diagnostic : si les deux courbes convergent, le modèle est bien équilibré. Si elles s'écartent, augmenter `min_samples_split` ou réduire `max_depth` renforcerait la régularisation. La croissance régulière du score de validation avec la taille du dataset suggère qu'un dataset plus grand améliorerait encore les performances.

<a id="5-conclusion"></a>
## 5. Conclusion — Justification du meilleur modèle

---

**Pourquoi Pipeline + RandomForest est la bonne architecture ?**

**1. Élimination structurelle du data leakage :**
L'usage d'un `sklearn.pipeline.Pipeline` garantit que chaque transformateur (SimpleImputer, StandardScaler, OneHotEncoder) n'est `fit` que sur les données d'entraînement du fold courant lors de la validation croisée. Il est architecturalement impossible de contaminer les données de validation avec des statistiques calculées sur celles-ci — contrairement à une approche manuelle où l'ordre des opérations doit être soigneusement contrôlé.

**2. Feature engineering pertinent et conservé :**
`Title`, `FamilySize`, `IsAlone` et `HasCabin` sont créées sur le DataFrame brut avant le split. Ces opérations sont purement déterministes (pas de statistiques apprises) et ne peuvent pas introduire de leakage. L'analyse des importances confirme la valeur prédictive de ces features construites manuellement.

**3. Pipeline unitaire, reproductible et déployable :**
`best_pipeline` est un objet sklearn sérialisable (joblib/pickle) qui encapsule l'intégralité de la chaîne de traitement. En production, un appel `best_pipeline.predict(X_new)` sur des données brutes (avec NaN, catégories textuelles) produit directement des prédictions sans étape intermédiaire manuelle — propriété fondamentale pour un déploiement fiable.

**4. Généralisation robuste :**
Le faible gap train/test et la convergence de la courbe d'apprentissage confirment que le modèle généralise bien sans surapprentissage. Le mécanisme de bagging du RandomForest, combiné à la contrainte de profondeur identifiée par GridSearchCV, joue efficacement son rôle régularisateur.

**En conclusion**, l'architecture Pipeline + ColumnTransformer + GridSearchCV représente la solution idéale : elle maximise simultanément la rigueur méthodologique (anti-leakage), la performance prédictive, la reproductibilité et la maintenabilité — les quatre critères décisifs pour un projet de machine learning académique et professionnel.